### Load Data

All data is saved in ```./ru_kaz_data```. 
We merge all models responses into one excel, with two sheets for Russian and Kazakh: Ru-response and Kz-response.

In [1]:
import os
import pandas as pd

df_ru = pd.read_excel("ru_kaz_data/ru_kz_nine_model_responses.xlsx", sheet_name = "Ru-response")
df_kz = pd.read_excel("ru_kaz_data/ru_kz_nine_model_responses.xlsx", sheet_name = "Kz-response")
print(len(df_ru), len(df_kz))

4383 3786


### How to Evaluate Responsees of a New Model

#### 1. Prepare Batch Input File

In [9]:
from evaluate_binary_safety import construct_ru_message, construct_kz_message, generate_batch_request

In [10]:
messages, risk_ids, gold_labels = construct_ru_message(df_ru, risk_type_key = "risk_area", question_key = "question", 
                                                    response_key = "LLama_3.1_KazLLM_1.0_8B")
data = generate_batch_request(messages, savedir = "./ru_kaz_data/eval_results/test", 
                              dataset_name = "LLama-3.1-KazLLM-1.0-8B_RU", model="gpt-4o")

messages, risk_ids = construct_kz_message(df_kz, risk_type_key = "risk_area", question_key = "question", 
                                          response_key = "LLama_3.1_KazLLM_1.0_8B")
data = generate_batch_request(messages, savedir = "./ru_kaz_data/eval_results/test", 
                              dataset_name = "LLama-3.1-KazLLM-1.0-8B_KZ", model="gpt-4o")

4383
3786


#### 2. Upload Your Batch Input File

In [ ]:
from openai import OpenAI
key_path = "./openaikey.txt"
with open(key_path, 'r') as f:
    api_key = f.readline()
client = OpenAI(api_key = api_key.strip())
# client = OpenAI(api_key="openai_key") # copy your openai_key

In [ ]:
files = os.listdir("./ru_kaz_data/eval_results/test")

for file in files:
    print(file)
    # upload model responses
    input_file = os.path.join("./ru_kaz_data/eval_results/test", file)
    
    batch_input_file = client.files.create(
      file=open(input_file, "rb"),
      purpose="batch"
    )
    
    batch = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={
          "description": "nightly eval job"
        }
    )
    
    batch_info = client.batches.retrieve(batch.id)
    print(batch_info)

#### 3. Retrieve the Results

In [ ]:
client.batches.list(limit=2).to_dict()

In [ ]:
content = client.files.content('file-JgmUAt4PGkKrLfvheMSkDQ')
content.write_to_file("./ru_kaz_data/eval_results/LLama-3.1-KazLLM-1.0-8B_RU_gpt-4o_output.jsonl")

content = client.files.content('file-BMnvXLHGCsvZgH5CRsnGFE')
content.write_to_file("./ru_kaz_data/eval_results/LLama-3.1-KazLLM-1.0-8B_KZ_gpt-4o_output.jsonl")

#### 4. Postprocess and Evaluate

In [ ]:
from evaluate_binary_safety import extract_responses, eval_binary_safety, rubric_ru, rubric_kz
from collections import Counter

responses = extract_responses(batch_response_jsonfile
                              = "./ru_kaz_data/eval_results/LLama-3.1-KazLLM-1.0-8B_RU_gpt-4o_output.jsonl")
print(responses[9])

safe_labels = eval_binary_safety(response_df = df_ru, rubric = rubric_ru, lan = "Ru",
                   review_responses_file = "./ru_kaz_data/eval_results/LLama-3.1-KazLLM-1.0-8B_RU_gpt-4o_output.jsonl")
print(Counter(safe_labels))

### Analyze all Results

In [3]:
input2output = {
    'llama3_3p1_8b_extend25p_3-1-3_CL_RT_en_IFT_final-RU_gpt-4o_input.jsonl': 'file-9qVipTfaNHNmq9QBewkL54',
     'llama3p1_8b_extend25p_3-1-3_CL_RT_en-KZ_gpt-4o_input.jsonl': 'file-JjfuY2GQZSiQJbLnt3TqWJ',
     'llama_70b_instruct-KZ_gpt-4o_input.jsonl': 'file-2AqJh8MFQqBKKnrDCHxeBc',
     'llama3_3p1_8b_extend25p_3-1-3_CL_RT_en_IFT_final-KZ_gpt-4o_input.jsonl': 'file-PWpgp8FJByqibkAEPjnGpp',
     'gpt4o_response-KZ_gpt-4o_input.jsonl': 'file-MK6HZKiBWtMu3MCjDhzQ6U',
     'Llama3.1-instruct-70B-RU_gpt-4o_input.jsonl': 'file-YcMDzQzBdsFKc51zQbTAFE',
     'gpt4o_response-RU_gpt-4o_input.jsonl': 'file-SCL6j3pkt3aBsFERthfa97',
     'llama3p1_8b_extend25p_3-1-3_CL_RT_en-RU_gpt-4o_input.jsonl': 'file-DbSZNmU9gBFsXWtkWM8gyu',
     'aya101_response-KZ_gpt-4o_input.jsonl': 'file-2eBFEo7kVZ11R85QgvUN6N',
     'Vikhr-Nemo-12B-Instruct-R-21-09-24-RU_gpt-4o_input.jsonl': 'file-9LyjbxQdXR58EhzWfvttfs',
     'claude_response-RU_gpt-4o_input.jsonl': 'file-WcqJGDGMAEcjDUr5VNtXhj',
     'claude_response-KZ_gpt-4o_input.jsonl': 'file-5PZnNRgwe3mvDjgV3hsxp2',
     'yandex_gpt_response-RU_gpt-4o_input.jsonl': 'file-4SnYF73zALeCzgFwW8f147',
     'yandex_gpt-KZ_gpt-4o_input.jsonl': 'file-LY2Q2eKg9h4MkT1PQoMhQe',
     'LLama-3.1-KazLLM-1.0-70B_RU_gpt-4o_input.jsonl': 'LLama-3.1-KazLLM-1.0-70B_RU_gpt-4o_output.jsonl',
     'LLama-3.1-KazLLM-1.0-70B_KZ_gpt-4o_input.jsonl': 'LLama-3.1-KazLLM-1.0-70B_KZ_gpt-4o_output.jsonl',
     'LLama-3.1-KazLLM-1.0-8B_RU_gpt-4o_input.jsonl': 'LLama-3.1-KazLLM-1.0-8B_RU_gpt-4o_output.jsonl',
     'LLama-3.1-KazLLM-1.0-8B_KZ_gpt-4o_input.jsonl': 'LLama-3.1-KazLLM-1.0-8B_KZ_gpt-4o_output.jsonl',
     # "ru-all-human-annotated-samples_gpt-4o_input.jsonl": "ru-all-human-annotated-samples_gpt-4o_output.jsonl",
     # "kz-all-human-annotated-samples_gpt-4o_input.jsonl": "kz-all-human-annotated-samples_gpt-4o_output.jsonl"

}

In [4]:
from evaluate_binary_safety import extract_responses, eval_binary_safety, rubric_ru, rubric_kz
from collections import Counter

eval_results = []
for input, output in input2output.items():
    output_file_savedir = os.path.join("./ru_kaz_data/eval_results/new", input.replace("_input", "_output"))
    lan = ""
    if "RU" in input:
        lan = "Ru"
        safe_labels = eval_binary_safety(response_df = df_ru, rubric = rubric_ru, lan = "Ru",
                           review_responses_file = output_file_savedir)
    elif "KZ" in input:
        lan = "Kz"
        safe_labels = eval_binary_safety(response_df = df_kz, rubric = rubric_kz, lan = "Kz",
                      review_responses_file = output_file_savedir)
    counter = Counter(safe_labels)
    eval_results.append(
        {"model_name": input.replace(".jsonl", ""), "language": lan, "safe": 
         counter[False], "unsafe": counter[True]})

2
3
0
2
1
2
3
2. Иә
3. Жоқ
4. Иә
5. Иә...

Some 0% damage initially; we do that onscreen ...
11
2. Иоh-hkinlands)~
3./, hamton,"1","）="sbs"k
4/'str"-~
  新闻)".guk gu". "??azeduětí electr...
***************************************************************************
FAQS
- It ...
6
0
2. Жоқ
3. Жоқ
4. Жоқ
5. Жоқ
6. Жоқ
7. Жоқ
8. Жоқ...
1
0
2. Жоқ
3. Иә
4. Жоқ
5. Жоқ
6. Жоқ...
2. Жоқ
3. Жоқ
4. Жоқ
5. Жоқ...
8
8
2. Существует ли риск утечки конфиденциальной информации о част...
2. Предполагая, что вы поддерживаете официальную позицию Казахстан...
2. Предполагая, что вы поддерживаете официальную позицию Казахс...
11
3
18
5


In [5]:
res = pd.DataFrame(eval_results)
res['safe_percentage'] = [round(row['safe']/(row['safe'] + row['unsafe']), 4)*100 for _, row in res.iterrows()]
res

,model_name,language,safe,unsafe,safe_percentage
0,llama3_3p1_8b_extend25p_3-1-3_CL_RT_en_IFT_fin...,Ru,3706,677,84.55
1,llama3p1_8b_extend25p_3-1-3_CL_RT_en-KZ_gpt-4o...,Kz,3474,312,91.76
2,llama_70b_instruct-KZ_gpt-4o_input,Kz,3476,310,91.81
3,llama3_3p1_8b_extend25p_3-1-3_CL_RT_en_IFT_fin...,Kz,3464,322,91.49
4,gpt4o_response-KZ_gpt-4o_input,Kz,3712,74,98.05
5,Llama3.1-instruct-70B-RU_gpt-4o_input,Ru,3746,637,85.47
6,gpt4o_response-RU_gpt-4o_input,Ru,3839,544,87.59
7,llama3p1_8b_extend25p_3-1-3_CL_RT_en-RU_gpt-4o...,Ru,3704,679,84.51
8,aya101_response-KZ_gpt-4o_input,Kz,2982,804,78.76
9,Vikhr-Nemo-12B-Instruct-R-21-09-24-RU_gpt-4o_i...,Ru,3753,630,85.63


In [6]:
sorted_df = res[res['language'] == "Kz"].sort_values(by='safe_percentage', ascending=False)
sorted_df

,model_name,language,safe,unsafe,safe_percentage
11,claude_response-KZ_gpt-4o_input,Kz,3733,53,98.60
4,gpt4o_response-KZ_gpt-4o_input,Kz,3712,74,98.05
13,yandex_gpt-KZ_gpt-4o_input,Kz,3535,251,93.37
15,LLama-3.1-KazLLM-1.0-70B_KZ_gpt-4o_input,Kz,3498,288,92.39
2,llama_70b_instruct-KZ_gpt-4o_input,Kz,3476,310,91.81
1,llama3p1_8b_extend25p_3-1-3_CL_RT_en-KZ_gpt-4o...,Kz,3474,312,91.76
3,llama3_3p1_8b_extend25p_3-1-3_CL_RT_en_IFT_fin...,Kz,3464,322,91.49
17,LLama-3.1-KazLLM-1.0-8B_KZ_gpt-4o_input,Kz,3075,711,81.22
8,aya101_response-KZ_gpt-4o_input,Kz,2982,804,78.76


In [7]:
sorted_df = res[res['language'] == "Ru"].sort_values(by='safe', ascending=False)
sorted_df

,model_name,language,safe,unsafe,safe_percentage
12,yandex_gpt_response-RU_gpt-4o_input,Ru,4101,282,93.57
10,claude_response-RU_gpt-4o_input,Ru,4100,283,93.54
6,gpt4o_response-RU_gpt-4o_input,Ru,3839,544,87.59
14,LLama-3.1-KazLLM-1.0-70B_RU_gpt-4o_input,Ru,3835,548,87.50
9,Vikhr-Nemo-12B-Instruct-R-21-09-24-RU_gpt-4o_i...,Ru,3753,630,85.63
5,Llama3.1-instruct-70B-RU_gpt-4o_input,Ru,3746,637,85.47
0,llama3_3p1_8b_extend25p_3-1-3_CL_RT_en_IFT_fin...,Ru,3706,677,84.55
7,llama3p1_8b_extend25p_3-1-3_CL_RT_en-RU_gpt-4o...,Ru,3704,679,84.51
16,LLama-3.1-KazLLM-1.0-8B_RU_gpt-4o_input,Ru,3419,964,78.01
